# Part 1: Supervised stopping-reason baseline

Predict the five stopping reasons using only the 157 training trials with visible labels. Each trial may have multiple reasons, so we fit one binary logistic regression per reason. The 1,014 unlabeled training trials are kept separately for later semi-supervised experiments.

We evaluate on the supplied 78-trial validation set. The main metric is **sample-wise F1**; we also report macro F1 and F1 for each reason. This experiment uses only the 150 provided reason embeddings.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(150)]

## Load the supplied training and validation sets

No new random split is needed. IDs identify trials and are excluded from the model inputs. The test data are reserved for later predictions.

In [2]:
# Support running from the repository root, AML-Midterm, or part1.
cwd = Path.cwd()
candidates = [cwd, cwd / "part1", cwd / "AML-Midterm" / "part1"]
DATA_DIR = next(
    (directory for directory in candidates
     if (directory / "part1_train.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Run from the repository root, AML-Midterm, or part1.")

train_df = pd.read_csv(DATA_DIR / "part1_train.csv")
val_df = pd.read_csv(DATA_DIR / "part1_val.csv")

assert train_df["ID"].is_unique and val_df["ID"].is_unique
assert set(train_df["ID"]).isdisjoint(val_df["ID"])
assert train_df[REASONS].isin([0, 1, -100]).all().all()
assert val_df[REASONS].isin([0, 1]).all().all()
for frame in (train_df, val_df):
    assert np.isfinite(frame[EMBEDDING_COLUMNS].to_numpy()).all()

## Keep trials with visible labels

A reason label of `0` is a known negative and `1` is a known positive. A value of `-100` means the label is hidden; it must not become a negative training target.

In [3]:
labeled_mask = train_df[REASONS].isin([0, 1]).all(axis=1)
unlabeled_mask = train_df[REASONS].eq(-100).all(axis=1)
assert (labeled_mask | unlabeled_mask).all(), "Unexpected partially labeled row."

labeled_train_df = train_df.loc[labeled_mask].copy()
unlabeled_train_df = train_df.loc[unlabeled_mask].copy()

X_train = labeled_train_df[EMBEDDING_COLUMNS].to_numpy()
y_train = labeled_train_df[REASONS].to_numpy(dtype=int)
X_val = val_df[EMBEDDING_COLUMNS].to_numpy()
y_val = val_df[REASONS].to_numpy(dtype=int)

# Every binary classifier needs both positive and negative examples.
assert ((y_train.sum(axis=0) > 0) & (y_train.sum(axis=0) < len(y_train))).all()

display(pd.DataFrame({
    "dataset": ["Labeled training", "Unlabeled training (reserved)", "Validation"],
    "trials": [len(labeled_train_df), len(unlabeled_train_df), len(val_df)],
    "used to fit this baseline": [True, False, False],
}))

,dataset,trials,used to fit this baseline
0,Labeled training,157,True
1,Unlabeled training (reserved),1014,False
2,Validation,78,False


## Standardize and fit logistic regression

The pipeline learns one mean and standard deviation per embedding column from the **157 labeled training trials only**. It reuses those statistics when predicting validation trials.

Each of the five classifiers uses L2-regularized logistic regression with `C=1`. We fix the probability threshold at 0.5 and do not tune settings on validation for this initial baseline. A trial can receive several reasons or none; no single-reason constraint is imposed.

In [4]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", OneVsRestClassifier(
        LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=5000,
            tol=1e-4,
            random_state=SEED,
        )
    )),
])

# A convergence failure should be addressed before trusting the results.
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    baseline_model.fit(X_train, y_train)

assert baseline_model.named_steps["scaler"].n_samples_seen_ == len(y_train)
print(f"Fitted five classifiers on {len(y_train)} labeled training trials.")

Fitted five classifiers on 157 labeled training trials.


## Evaluate on the same validation set used for future comparisons

Sample-wise F1 computes F1 for each trial's set of reasons and averages across trials. Macro F1 averages the five per-reason F1 scores. Training scores help reveal overfitting; validation scores measure performance on held-out trials. Undefined F1 values are assigned zero.

In [5]:
THRESHOLD = 0.5
train_probabilities = baseline_model.predict_proba(X_train)
val_probabilities = baseline_model.predict_proba(X_val)
train_predictions = (train_probabilities >= THRESHOLD).astype(int)
val_predictions = (val_probabilities >= THRESHOLD).astype(int)

for probabilities, labels in ((train_probabilities, y_train), (val_probabilities, y_val)):
    assert probabilities.shape == labels.shape
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

score_rows = []
for split, actual, predicted in (
    ("Training", y_train, train_predictions),
    ("Validation", y_val, val_predictions),
):
    score_rows.append({
        "split": split,
        "sample-wise F1": f1_score(actual, predicted, average="samples", zero_division=0),
        "macro F1": f1_score(actual, predicted, average="macro", zero_division=0),
    })
baseline_scores = pd.DataFrame(score_rows).set_index("split")
display(baseline_scores.round(4))

per_reason_scores = pd.DataFrame({
    "reason": REASONS,
    "training positives": y_train.sum(axis=0),
    "validation positives": y_val.sum(axis=0),
    "validation predicted positives": val_predictions.sum(axis=0),
    "validation F1": f1_score(y_val, val_predictions, average=None, zero_division=0),
}).set_index("reason")
display(per_reason_scores.round(4))

print(f"Validation trials with no predicted reason: {(val_predictions.sum(axis=1) == 0).sum()}")
print(f"Validation sample-wise F1: {baseline_scores.loc['Validation', 'sample-wise F1']:.4f}")

,sample-wise F1,macro F1
split,,
Training,1.0000,1.0000
Validation,0.8162,0.6685


,training positives,validation positives,validation predicted positives,validation F1
reason,,,,
Safety,16,9,6,0.8000
Funding,8,4,1,0.4000
Efficacy,17,14,8,0.6364
Recruitment,101,50,50,0.9800
Other,25,10,9,0.5263


Validation trials with no predicted reason: 8
Validation sample-wise F1: 0.8162


## Compare with predicting Recruitment for every trial

This simple reference helps show whether the model adds value beyond the most common stopping reason. Both methods are evaluated on the identical validation trials.

In [6]:
recruitment_predictions = np.zeros_like(y_val)
recruitment_predictions[:, REASONS.index("Recruitment")] = 1

reference_comparison = pd.DataFrame([
    {
        "method": "Always Recruitment",
        "validation sample-wise F1": f1_score(
            y_val, recruitment_predictions, average="samples", zero_division=0
        ),
        "validation macro F1": f1_score(
            y_val, recruitment_predictions, average="macro", zero_division=0
        ),
    },
    {
        "method": "Labeled-only logistic regression",
        "validation sample-wise F1": baseline_scores.loc["Validation", "sample-wise F1"],
        "validation macro F1": baseline_scores.loc["Validation", "macro F1"],
    },
]).set_index("method")
display(reference_comparison.round(4))

,validation sample-wise F1,validation macro F1
method,,
Always Recruitment,0.6239,0.1562
Labeled-only logistic regression,0.8162,0.6685


## Baseline results

On the 78 validation trials, labeled-only logistic regression achieves **sample-wise F1 = 0.8162** and **macro F1 = 0.6685**. Always predicting Recruitment achieves sample-wise F1 = 0.6239, so logistic regression improves this reference by 0.1923 (19.23 percentage points).

Per-reason validation F1 is 0.8000 for Safety, 0.4000 for Funding, 0.6364 for Efficacy, 0.9800 for Recruitment, and 0.5263 for Other. Funding has only four validation positives, so a few changed predictions can substantially change its score. Eight validation trials receive no predicted reason at the fixed 0.5 threshold.

Training F1 is 1.0000 for both sample-wise and macro averages. The model fits the small labeled training set perfectly but makes held-out errors, so training scores alone cannot guide model selection. These results are a fixed, untuned validation baseline, not a test or Kaggle score.

## Compare L2 regularization strengths

Compare the prespecified values `C = 0.01, 0.1, 1, 10`. Smaller `C` means stronger regularization. Keep the same 157 labeled training trials, 78 validation trials, embedding features, training-only standardization, and 0.5 probability threshold.

The original `C=1` baseline remains unchanged. Choose the supervised comparison model by validation sample-wise F1; exact ties favor smaller `C`. Also inspect macro F1 and each reason's F1. Because this step uses validation performance to select `C`, its winning score is a tuning result rather than an independent test estimate.

In [7]:
from sklearn.base import clone

C_VALUES = [0.01, 0.1, 1.0, 10.0]
regularization_models = {}
regularization_val_predictions = {}
regularization_rows = []
regularization_reason_rows = []

for C in C_VALUES:
    if C == 1.0:
        # Reuse the fitted original baseline without changing it.
        model = baseline_model
    else:
        model = clone(baseline_model).set_params(classifier__estimator__C=C)
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            model.fit(X_train, y_train)

    assert model.named_steps["scaler"].n_samples_seen_ == len(y_train)
    train_pred = (model.predict_proba(X_train) >= THRESHOLD).astype(int)
    val_pred = (model.predict_proba(X_val) >= THRESHOLD).astype(int)
    regularization_models[C] = model
    regularization_val_predictions[C] = val_pred
    regularization_rows.append({
        "C": C,
        "training sample-wise F1": f1_score(
            y_train, train_pred, average="samples", zero_division=0
        ),
        "validation sample-wise F1": f1_score(
            y_val, val_pred, average="samples", zero_division=0
        ),
        "validation macro F1": f1_score(
            y_val, val_pred, average="macro", zero_division=0
        ),
        "validation trials with no predicted reason": int((val_pred.sum(axis=1) == 0).sum()),
    })
    regularization_reason_rows.append({
        "C": C,
        **dict(zip(REASONS, f1_score(y_val, val_pred, average=None, zero_division=0))),
    })

regularization_results = pd.DataFrame(regularization_rows).set_index("C")
regularization_reason_f1 = pd.DataFrame(regularization_reason_rows).set_index("C")
print("Overall F1 comparison:")
display(regularization_results.round(4))
print("Per-reason validation F1:")
display(regularization_reason_f1.round(4))

selected_C = float(
    regularization_results.reset_index().sort_values(
        ["validation sample-wise F1", "C"],
        ascending=[False, True],
        kind="stable",
    ).iloc[0]["C"]
)
selected_supervised_model = regularization_models[selected_C]
selected_validation_f1 = regularization_results.loc[selected_C, "validation sample-wise F1"]
original_validation_f1 = baseline_scores.loc["Validation", "sample-wise F1"]
print(f"Selected C: {selected_C:g}")
print(f"Selected validation sample-wise F1: {selected_validation_f1:.4f}")
print(f"Change from original C=1 baseline: {selected_validation_f1 - original_validation_f1:+.4f}")

Overall F1 comparison:


,training sample-wise F1,validation sample-wise F1,validation macro F1,validation trials with no predicted reason
C,,,,
0.01,0.8705,0.6880,0.3641,22
0.10,0.9979,0.7991,0.5829,13
1.00,1.0000,0.8162,0.6685,8
10.00,1.0000,0.8248,0.6804,8


Per-reason validation F1:


,Safety,Funding,Efficacy,Recruitment,Other
C,,,,,
0.01,0.3636,0.0,0.1333,0.9901,0.3333
0.10,0.7143,0.0,0.6667,1.0000,0.5333
1.00,0.8000,0.4,0.6364,0.9800,0.5263
10.00,0.8000,0.4,0.6957,0.9800,0.5263


Selected C: 10
Selected validation sample-wise F1: 0.8248
Change from original C=1 baseline: +0.0085


## Regularization comparison results

Among the four prespecified settings, `C=10` has the highest validation sample-wise F1: **0.8248**, compared with **0.8162** for the original `C=1` baseline. Validation macro F1 increases from 0.6685 to 0.6804. The sample-wise improvement is 0.0085, or approximately 0.85 percentage points. With only 78 validation trials, this small gain should not be treated as proof of better test performance.

Stronger regularization did not improve this comparison: `C=0.1` achieves sample-wise F1 of 0.7991 and `C=0.01` achieves 0.6880. Both fail to predict any Funding positives at the fixed threshold. At `C=0.01`, the model predicts no reason for 22 validation trials, compared with eight for `C=1` and `C=10`; this is consistent with overly restrictive weights suppressing positive predictions, particularly for rare reasons.

The selected `C=10` model uses weaker regularization. Its training sample-wise F1 remains 1.0000, while Efficacy validation F1 improves from 0.6364 to 0.6957 and the other four per-reason F1 scores remain unchanged. Perfect training F1 alone therefore did not establish that stronger regularization would improve generalization. Keep the original baseline and the tuned supervised result for subsequent comparisons.

## One follow-up check: C=100

After the initial four-value comparison selected `C=10` at the upper edge of the grid, test `C=100` once. This follow-up was chosen after inspecting the earlier results; it was not part of the original prespecified grid. Keep the data, preprocessing, threshold, and other model settings fixed. Compare with `C=10`, and select by sample-wise validation F1 with ties favoring smaller `C`.

In [8]:
followup_C = 100.0
followup_model = clone(baseline_model).set_params(classifier__estimator__C=followup_C)
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    followup_model.fit(X_train, y_train)

assert followup_model.named_steps["scaler"].n_samples_seen_ == len(y_train)
followup_train_predictions = (followup_model.predict_proba(X_train) >= THRESHOLD).astype(int)
followup_val_predictions = (followup_model.predict_proba(X_val) >= THRESHOLD).astype(int)
regularization_models[followup_C] = followup_model
regularization_val_predictions[followup_C] = followup_val_predictions
regularization_results.loc[followup_C] = {
    "training sample-wise F1": f1_score(
        y_train, followup_train_predictions, average="samples", zero_division=0
    ),
    "validation sample-wise F1": f1_score(
        y_val, followup_val_predictions, average="samples", zero_division=0
    ),
    "validation macro F1": f1_score(
        y_val, followup_val_predictions, average="macro", zero_division=0
    ),
    "validation trials with no predicted reason": int((followup_val_predictions.sum(axis=1) == 0).sum()),
}
regularization_reason_f1.loc[followup_C] = f1_score(
    y_val, followup_val_predictions, average=None, zero_division=0
)

print("C=10 versus C=100:")
display(regularization_results.loc[[10.0, followup_C]].round(4))
print("Per-reason validation F1:")
display(regularization_reason_f1.loc[[10.0, followup_C]].round(4))

selected_C = float(
    regularization_results.reset_index().sort_values(
        ["validation sample-wise F1", "C"],
        ascending=[False, True],
        kind="stable",
    ).iloc[0]["C"]
)
selected_supervised_model = regularization_models[selected_C]
selected_validation_f1 = regularization_results.loc[selected_C, "validation sample-wise F1"]
followup_change = (
    regularization_results.loc[followup_C, "validation sample-wise F1"]
    - regularization_results.loc[10.0, "validation sample-wise F1"]
)
print(f"Change from C=10 to C=100: {followup_change:+.4f}")
print(f"Validation trials with any changed label prediction: {(followup_val_predictions != regularization_val_predictions[10.0]).any(axis=1).sum()}")
print(f"Selected C after the follow-up: {selected_C:g}")
print(f"Selected validation sample-wise F1: {selected_validation_f1:.4f}")

C=10 versus C=100:


,training sample-wise F1,validation sample-wise F1,validation macro F1,validation trials with no predicted reason
C,,,,
10.0,1.0,0.8248,0.6804,8
100.0,1.0,0.8462,0.7677,8


Per-reason validation F1:


,Safety,Funding,Efficacy,Recruitment,Other
C,,,,,
10.0,0.8000,0.4000,0.6957,0.9800,0.5263
100.0,0.9412,0.6667,0.6957,0.9796,0.5556


Change from C=10 to C=100: +0.0214
Validation trials with any changed label prediction: 8
Selected C after the follow-up: 100
Selected validation sample-wise F1: 0.8462


## Follow-up result

`C=100` improves validation sample-wise F1 from **0.8248 to 0.8462** (approximately 2.14 percentage points) and macro F1 from **0.6804 to 0.7677**. Training sample-wise F1 remains 1.0000. Eight validation trials have at least one changed predicted label relative to `C=10`.

Safety F1 increases from 0.8000 to 0.9412, Funding from 0.4000 to 0.6667, and Other from 0.5263 to 0.5556. Efficacy remains 0.6957, while Recruitment decreases slightly from 0.9800 to 0.9796. Both models predict no reason for eight validation trials.

After this check, `C=100` is the selected supervised comparator. The final `C=1000` check is recorded separately below. Keep the original `C=1` baseline for reference. This selection used the same small validation set, so the gain is a validation tuning result rather than an independently confirmed test improvement.

## Final regularization check: C=1000

After the improvement at `C=100`, test one final weaker regularization setting, `C=1000`. This is a second adaptive follow-up to the original four-value grid. Use the same 157 labeled training trials, 78 validation trials, training-only standardization, and fixed 0.5 threshold. Select by validation sample-wise F1; ties favor smaller `C`. Stop this search after this check.

In [9]:
final_check_C = 1000.0
final_check_model = clone(baseline_model).set_params(classifier__estimator__C=final_check_C)
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    final_check_model.fit(X_train, y_train)

assert final_check_model.named_steps["scaler"].n_samples_seen_ == len(y_train)
final_check_train_predictions = (final_check_model.predict_proba(X_train) >= THRESHOLD).astype(int)
final_check_val_predictions = (final_check_model.predict_proba(X_val) >= THRESHOLD).astype(int)
regularization_models[final_check_C] = final_check_model
regularization_val_predictions[final_check_C] = final_check_val_predictions
regularization_results.loc[final_check_C] = {
    "training sample-wise F1": f1_score(
        y_train, final_check_train_predictions, average="samples", zero_division=0
    ),
    "validation sample-wise F1": f1_score(
        y_val, final_check_val_predictions, average="samples", zero_division=0
    ),
    "validation macro F1": f1_score(
        y_val, final_check_val_predictions, average="macro", zero_division=0
    ),
    "validation trials with no predicted reason": int((final_check_val_predictions.sum(axis=1) == 0).sum()),
}
regularization_reason_f1.loc[final_check_C] = f1_score(
    y_val, final_check_val_predictions, average=None, zero_division=0
)

print("C=100 versus C=1000:")
display(regularization_results.loc[[100.0, final_check_C]].round(4))
print("Per-reason validation F1:")
display(regularization_reason_f1.loc[[100.0, final_check_C]].round(4))

selected_C = float(
    regularization_results.reset_index().sort_values(
        ["validation sample-wise F1", "C"],
        ascending=[False, True],
        kind="stable",
    ).iloc[0]["C"]
)
selected_supervised_model = regularization_models[selected_C]
selected_validation_f1 = regularization_results.loc[selected_C, "validation sample-wise F1"]
final_check_change = (
    regularization_results.loc[final_check_C, "validation sample-wise F1"]
    - regularization_results.loc[100.0, "validation sample-wise F1"]
)
print(f"Change from C=100 to C=1000: {final_check_change:+.4f}")
print(f"Validation trials with any changed label prediction: {(final_check_val_predictions != regularization_val_predictions[100.0]).any(axis=1).sum()}")
print(f"Selected C after the final check: {selected_C:g}")
print(f"Selected validation sample-wise F1: {selected_validation_f1:.4f}")

C=100 versus C=1000:


,training sample-wise F1,validation sample-wise F1,validation macro F1,validation trials with no predicted reason
C,,,,
100.0,1.0,0.8462,0.7677,8
1000.0,1.0,0.8803,0.8133,4


Per-reason validation F1:


,Safety,Funding,Efficacy,Recruitment,Other
C,,,,,
100.0,0.9412,0.6667,0.6957,0.9796,0.5556
1000.0,0.9412,0.6667,0.8571,0.9697,0.6316


Change from C=100 to C=1000: +0.0342
Validation trials with any changed label prediction: 7
Selected C after the final check: 1000
Selected validation sample-wise F1: 0.8803


## Final regularization result

`C=1000` improves validation sample-wise F1 from **0.8462 to 0.8803** (approximately 3.42 percentage points) and macro F1 from **0.7677 to 0.8133**. Training sample-wise F1 remains 1.0000. Seven validation trials have at least one changed predicted label relative to `C=100`; the number of trials with no predicted reason falls from eight to four.

Efficacy validation F1 increases from 0.6957 to 0.8571 and Other from 0.5556 to 0.6316. Safety stays at 0.9412 and Funding at 0.6667, while Recruitment decreases from 0.9796 to 0.9697.

Select **`C=1000`** as the tuned supervised comparator and end this regularization search. Preserve the original `C=1` baseline (sample-wise F1 0.8162). All settings used the same labeled training trials, training-only standardization, and fixed 0.5 prediction threshold. The selected score used the same 78 validation trials for tuning, so it does not establish performance on an independent test set.

## Next comparison

Keep both the original `C=1` baseline and the model selected in the regularization comparison and its two follow-ups (`C=100` and `C=1000`) as supervised references when evaluating semi-supervised methods that use the 1,014 unlabeled training trials. Use the same validation trials and metrics, and report the tuning performed for each method.

The experiments so far use only reason embeddings and a fixed 0.5 probability threshold. Additional features and generated labels remain later steps. The supervised test submission is exported below, with an explicitly documented rule for empty prediction sets. Rare categories have few validation examples, so small changes need cautious interpretation.

## Export the supervised Kaggle baseline

Use the fitted **`C=1000`** supervised pipeline, trained on the original 157 labeled training trials. Reuse its training means and standard deviations for the test embeddings. Do not retrain on validation or unlabeled trials.

Keep the 0.5 prediction threshold. For submission, if no reason exceeds the threshold, choose the highest-probability reason instead: every Part 1 trial has at least one stopping reason, and this ensures a nonempty reason string in every CSV row. This is an explicit postprocessing change from the earlier threshold-only baseline. Report its validation score separately so the Kaggle baseline and later comparisons use the same prediction rule.

Write one row per test trial with columns `ID` and `Predicted`, using space-separated reason names in the sample submission's ID order. This cell creates the CSV locally; upload it manually to the Part 1 competition.

In [10]:
def submission_reason_predictions(probabilities):
    """Threshold each reason; use the most probable reason for empty sets."""
    predicted = (probabilities >= THRESHOLD).astype(int)
    needs_reason = predicted.sum(axis=1) == 0
    rows = np.flatnonzero(needs_reason)
    predicted[rows, probabilities[rows].argmax(axis=1)] = 1
    return predicted, needs_reason


submission_model = regularization_models[1000.0]
assert submission_model.named_steps["classifier"].estimator.C == 1000.0
assert submission_model.named_steps["scaler"].n_samples_seen_ == len(y_train)

# Score the exact submission rule on validation before exporting test labels.
submission_val_probabilities = submission_model.predict_proba(X_val)
submission_val_predictions, val_needs_reason = submission_reason_predictions(
    submission_val_probabilities
)
submission_validation_f1 = f1_score(
    y_val, submission_val_predictions, average="samples", zero_division=0
)
submission_validation_macro_f1 = f1_score(
    y_val, submission_val_predictions, average="macro", zero_division=0
)
display(pd.DataFrame([
    {
        "prediction rule": "C=1000, threshold only",
        "validation sample-wise F1": regularization_results.loc[1000.0, "validation sample-wise F1"],
        "validation macro F1": regularization_results.loc[1000.0, "validation macro F1"],
    },
    {
        "prediction rule": "C=1000, threshold plus nonempty fallback (submitted)",
        "validation sample-wise F1": submission_validation_f1,
        "validation macro F1": submission_validation_macro_f1,
    },
]).set_index("prediction rule").round(4))
print(f"Validation trials requiring fallback: {val_needs_reason.sum()}")

test_df = pd.read_csv(DATA_DIR / "part1_test.csv")
sample_submission = pd.read_csv(DATA_DIR / "part1_sample_submission.csv")
assert test_df["ID"].is_unique and sample_submission["ID"].is_unique
assert set(test_df["ID"]) == set(sample_submission["ID"])
assert set(test_df["ID"]).isdisjoint(train_df["ID"])
assert set(test_df["ID"]).isdisjoint(val_df["ID"])
assert np.isfinite(test_df[EMBEDDING_COLUMNS].to_numpy()).all()

test_probabilities = submission_model.predict_proba(test_df[EMBEDDING_COLUMNS].to_numpy())
assert test_probabilities.shape == (len(test_df), len(REASONS))
assert np.isfinite(test_probabilities).all()
assert ((test_probabilities >= 0) & (test_probabilities <= 1)).all()
test_predictions, test_needs_reason = submission_reason_predictions(test_probabilities)

reason_strings = [
    " ".join(reason for reason, present in zip(REASONS, row) if present)
    for row in test_predictions
]
submission = pd.DataFrame({"ID": test_df["ID"], "Predicted": reason_strings})
submission = submission.set_index("ID").loc[sample_submission["ID"]].reset_index()
assert submission["ID"].tolist() == sample_submission["ID"].tolist()
assert submission.columns.tolist() == ["ID", "Predicted"]
assert submission["Predicted"].str.len().gt(0).all()
assert submission["Predicted"].map(
    lambda value: set(value.split()).issubset(REASONS)
).all()

submission_path = DATA_DIR / "part1_logistic_regression_C1000_submission.csv"
submission.to_csv(submission_path, index=False)
saved_submission = pd.read_csv(submission_path)
pd.testing.assert_frame_equal(saved_submission, submission)
print(f"Saved {len(submission)} test predictions to: {submission_path.resolve()}")
print(f"Test trials requiring fallback: {test_needs_reason.sum()}")
display(submission.head(10))
display(pd.Series(test_predictions.sum(axis=0), index=REASONS, name="test predicted positives"))

,validation sample-wise F1,validation macro F1
prediction rule,,
"C=1000, threshold only",0.8803,0.8133
"C=1000, threshold plus nonempty fallback (submitted)",0.8932,0.8043


Validation trials requiring fallback: 4
Saved 312 test predictions to: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/part1_logistic_regression_C1000_submission.csv
Test trials requiring fallback: 16


,ID,Predicted
0,TR00756,Efficacy
1,TR05267,Safety Other
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Efficacy
5,TR01460,Recruitment
6,TR00199,Funding Recruitment
7,TR09959,Recruitment
8,TR01886,Recruitment
9,TR09553,Efficacy Other


Safety          58
Funding          7
Efficacy        57
Recruitment    179
Other           63
Name: test predicted positives, dtype: int64

## Recorded supervised baseline for future comparisons

This is the supervised baseline submitted to Kaggle. All regularization experiments, validation outputs, and the reproducible CSV export are retained above.

| Setting | Recorded value |
|---|---|
| Features | `emb_0` through `emb_149` only |
| Training data | 157 trials with complete reason labels |
| Unlabeled data | 1,014 trials reserved; unused in this model |
| Validation data | The supplied 78 labeled trials |
| Preprocessing | StandardScaler fitted on the 157 labeled training trials |
| Classifier | Five one-vs-rest L2 logistic regressions |
| Selected regularization | `C=1000` |
| Other settings | `solver="lbfgs"`, `max_iter=5000`, `tol=1e-4`, `random_state=42` |
| Selection history | Initial `C=0.01, 0.1, 1, 10`; adaptive follow-ups `100` and `1000` |
| Prediction rule | Select every probability >= 0.5; if none qualify, select the highest-probability reason |
| Fallback cases | 4 validation trials; 16 test trials |
| Validation sample-wise F1, submitted rule | **0.8932** |
| Validation macro F1, submitted rule | **0.8043** |
| Validation sample-wise / macro F1, threshold-only rule | 0.8803 / 0.8133 |
| Training sample-wise / macro F1 | 1.0000 / 1.0000 |
| Kaggle leaderboard score | **0.87072** |
| Submission file | `part1_logistic_regression_C1000_submission.csv` |
| Test predictions | 312 rows, columns `ID` and `Predicted` |

**Kaggle provenance:** Michael reported the score after manually uploading this submission. The leaderboard snapshot was rank **32**, with the leading score **0.92735**. These values were reported in the conversation, not independently retrieved; rank and leading score can change. The validation-to-Kaggle sample-wise F1 difference is approximately 0.0224 (2.24 percentage points, using the unrounded validation score).

**Comparison protocol:** evaluate future semi-supervised methods on the same 78 validation trials, using the same nonempty-label prediction rule, sample-wise F1, macro F1, and per-reason F1. Compare their Kaggle scores with **0.87072** and document any tuning or changes in training data. The supervised model was not retrained on validation for this submission. Validation results were used for model selection.

Submission SHA-256: `bbd05ba89d4ac0a21864bbdd029a21633f2ee3ead391f25eda3148d73a276c10`.
